# clue_grid

> Phase 11 grids: the rule-agent sweep and its frozen settings rule, the incentive check, the live grids (pilot, main, framing, liar, and the optional matched-effort and scale arms), the numpy counterpart of the main grid, quick cells, and `CLUE_PLAN` with its hash.

In [ ]:
#| default_exp clue_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json

import numpy as np

from explRL.cluetask import CLUE_PAIRS
from explRL.llm_budget import PRICES
from explRL.llm_grid import HAIKU, SONNET, llmCell

CHANNELS = ('none', 'blackboard', 'messages', 'hierarchical')
CLUE_SETTINGS = dict(m=2, r=1, R=30, guess=0.0, value=(1, 2, 3), life=30, penalty=0.5,     # chosen by the sweep (T5)
                     effort='low', maxTokens=8000)
BOARD = ('m', 'r', 'R', 'value', 'life', 'penalty')
CLUE_PLAN = dict(
    version=1, frozen=True, model=HAIKU, settings=CLUE_SETTINGS, prices=PRICES,
    seeds=dict(sweep=[800, 819], pilot=[1100, 1101], main=[900, 904], framing=[910, 912], liar=[900, 902],
               effort=[900, 902], scale=[920, 922]),
    analyses=dict(
        main='welfare per agent-turn and team total vs N per channel (seeds 900-904)',
        benefit='per (channel, N): welfare minus the none cell, paired on seed, bootstrap CI',
        information='shares per held clue, redundant shares, assisted solves, share-to-use time per (channel, N)',
        numpy="Spearman rho over the 12 (channel, N) cells, LLM vs rule agents ('needed'), bootstrap CI over seeds",
        framing='blackboard, N 4 and 8, seeds 910-912: team minus own framing on sharePerHeld and welfare, paired',
        liar='N = 4, seeds 900-902: honest welfare loss from one false-clue liar on the blackboard and with messages; '
             'blackboard-minus-messages sign vs H3',
        optional='effort: (Sonnet - Haiku) blackboard benefit at N = 4, effort medium; scale: N = 16'),
    nboot=2000, ci=0.95, exploratory=True)


def clueHash(plan=CLUE_PLAN):
    return hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest()[:12]


def _board(s=CLUE_SETTINGS):
    return {k: s[k] for k in BOARD}


def clueCell(study, level, N, seed, runner='llm', s=CLUE_SETTINGS, world=None, **cfg):
    kw = dict(task='clue', turns=40, effort=s['effort'], maxTokens=s['maxTokens'], world=world or _board(s))
    return llmCell(study, study, level, N, seed, runner, **{**kw, **cfg})


def clueSweep(pairs=CLUE_PAIRS, sizes=(1, 2, 4, 8, 16, 32, 64), seeds=range(800, 820)):
    "Part B: rule agents over (m, R) x r x guess x share x N x channel (study clueSweep)."
    return [clueCell('clueSweep', f'm={m}|R={R}|r={r}|g={g}|{sh}', N, sd, 'numpy', channel=ch,
                     world=dict(m=m, R=R, r=r, guess=g, share=sh, value=(1, 2, 3), life=30, penalty=0.5))
            for m, R in pairs for r in (1, 2) for g in (0.0, 0.2) for sh in ('needed', 'never')
            for N in sizes for ch in CHANNELS for sd in seeds]


def _at4(rows, share='needed'):
    out = {}
    for r in rows:
        if r['study'] == 'clueSweep' and int(float(r['N'])) == 4 and str(r['level']).endswith('|' + share):
            out.setdefault(str(r['level']).rsplit('|', 1)[0], {}).setdefault(r['channel'], []).append(r)
    return out


def chooseClueSettings(rows):
    """The frozen rule (FR-B3), at N = 4 with 'needed' sharing: blackboard welfare at most 0.8 of the pooled bound and
    no-channel welfare at most half the blackboard's; the largest gap relative to the bound; ties: smaller R, then m."""
    ok = {}
    for lv, by in _at4(rows).items():
        bb, no = (float(np.mean([float(x['pooledShare']) for x in by.get(c, [])])) for c in ('blackboard', 'none'))
        if bb <= 0.8 and no <= 0.5 * bb:
            ok[lv] = bb - no
    if not ok:
        return None
    parse = lambda lv: {k: float(v) if k == 'g' else int(v) for k, v in (x.split('=') for x in lv.split('|'))}
    best = max(ok, key=lambda lv: (ok[lv], -parse(lv)['R'], -parse(lv)['m']))
    p = parse(best)
    return dict(m=p['m'], r=p['r'], R=p['R'], guess=p['g'])


def sweepIncentives(rows, s):
    "At N = 4 with the blackboard and the chosen settings: team welfare needed vs never; sharer vs non-sharer reward."
    lv = f"m={s['m']}|R={s['R']}|r={s['r']}|g={s['guess']}"
    get = lambda sh: _at4(rows, sh).get(lv, {}).get('blackboard', [])
    m = lambda rs, k: float(np.nanmean([float(x[k]) for x in rs])) if rs else float('nan')
    return dict(welfareNeeded=m(get('needed'), 'welfare'), welfareNever=m(get('never'), 'welfare'),
                sharerReward=m(get('needed'), 'sharerReward'), nonSharerReward=m(get('needed'), 'nonSharerReward'))


def cluePilotGrid(s=CLUE_SETTINGS):
    return [clueCell('cluePilot', 'pilot', N, sd, s=s, channel=ch, turns=30)
            for ch, N in (('blackboard', 4), ('none', 4), ('none', 1)) for sd in (1100, 1101)]


def clueMainGrid(s=CLUE_SETTINGS):
    return [clueCell('clue', 'clue', N, sd, s=s, channel=ch) for ch in CHANNELS for N in (2, 4, 8) for sd in range(900, 905)]


def clueNumpyGrid(s=CLUE_SETTINGS):
    "The main cells with rule agents ('needed' sharing, the chosen guess rate)."
    return [clueCell('clueNumpy', 'clue', N, sd, 'numpy', s=s, channel=ch, world=dict(_board(s), guess=s['guess'], share='needed'))
            for ch in CHANNELS for N in (2, 4, 8) for sd in range(900, 905)]


def clueFramingGrid(s=CLUE_SETTINGS):
    return [clueCell('clueFraming', f'framing={f}', N, sd, s=s, channel='blackboard', framing=f)
            for f in ('own', 'team') for N in (4, 8) for sd in range(910, 913)]


def clueLiarGrid(s=CLUE_SETTINGS):
    return [clueCell('clueLiar', 'liar', 4, sd, s=s, channel=ch, nLiar=1) for ch in ('blackboard', 'messages')
            for sd in range(900, 903)]


def clueEffortGrid(s=CLUE_SETTINGS):
    return [clueCell('clueEffort', f'{m.split("-")[1]}|medium', 4, sd, s=s, channel=ch, model=m, effort='medium')
            for m in (HAIKU, SONNET) for ch in ('none', 'blackboard') for sd in range(900, 903)]


def clueScaleGrid(s=CLUE_SETTINGS):
    return [clueCell('clueScale', 'scale', 16, sd, s=s, channel=ch) for ch in ('none', 'blackboard') for sd in range(920, 923)]


CLUE_GRIDS = dict(main=clueMainGrid, framing=clueFramingGrid, liar=clueLiarGrid)        # priority order (K6)
CLUE_OPTIONAL = dict(effort=clueEffortGrid, scale=clueScaleGrid)


def clueQuick():
    "One cell of each grid at 3 turns (numpy: 10 turns, a few sizes), for the fake-client pipeline check."
    cells = [g()[0] for g in (cluePilotGrid, clueMainGrid, clueFramingGrid, clueLiarGrid, clueEffortGrid, clueScaleGrid)]
    cells = [dict(c, cfg=dict(c['cfg'], turns=3)) for c in cells]
    numpy = [c for c in clueNumpyGrid() if c['seed'] == 900 and c['N'] in (2, 4)] + \
        [c for c in clueSweep(sizes=(4,), seeds=(800,)) if c['level'].startswith('m=3|R=105|r=1|g=0.0')]
    return cells + [dict(c, cfg=dict(c['cfg'], turns=10)) for c in numpy]

## Tests

In [ ]:
from explRL.clue_grid import *
from explRL.llm_grid import estimateGrid, HAIKU, SONNET
from explRL.bench_grid import rowKey
from explRL.llm_runs import LLMConfig, cellRow

calls = lambda cells, m=None: sum(v['calls'] for k, v in estimateGrid(cells).items() if m in (None, k))
# grid sizes and call counts (design-low-level section 6)
assert len(clueSweep()) == 5 * 2 * 2 * 2 * 7 * 4 * 20 == 22400 and estimateGrid(clueSweep()) == {}
assert calls(cluePilotGrid()) == 2 * (4 + 4 + 1) * 30 and calls(clueMainGrid()) == 11200
assert calls(clueFramingGrid()) == 2880 and calls(clueLiarGrid()) == 720
assert calls(clueEffortGrid(), HAIKU) == calls(clueEffortGrid(), SONNET) == 960 and calls(clueScaleGrid()) == 3840
assert len(clueNumpyGrid()) == 60 and estimateGrid(clueNumpyGrid()) == {}
cells = cluePilotGrid() + clueMainGrid() + clueFramingGrid() + clueLiarGrid() + clueEffortGrid() + clueScaleGrid() + clueNumpyGrid()
keys = [rowKey(cellRow(c, LLMConfig.fromDict(c['cfg']), dict(episode=0))) for c in cells]
assert len(set(keys)) == len(keys) and all(c['cfg']['task'] == 'clue' for c in cells)
assert all(c['cfg']['world'] == {k: CLUE_SETTINGS[k] for k in BOARD} for c in clueMainGrid())
assert {c['seed'] for c in clueMainGrid()} == set(range(900, 905)) and {c['seed'] for c in clueLiarGrid()} == {900, 901, 902}

# the frozen settings rule on synthetic sweep rows
def row(m, R, r, g, share, ch, ps, N=4, **kw):
    return dict(study='clueSweep', level=f'm={m}|R={R}|r={r}|g={g}|{share}', N=N, channel=ch, pooledShare=ps, **kw)
rows = [row(3, 105, 1, 0.0, 'needed', 'blackboard', 0.6), row(3, 105, 1, 0.0, 'needed', 'none', 0.1),     # gap 0.5
        row(2, 30, 1, 0.0, 'needed', 'blackboard', 0.9), row(2, 30, 1, 0.0, 'needed', 'none', 0.0),       # too easy
        row(3, 210, 2, 0.2, 'needed', 'blackboard', 0.5), row(3, 210, 2, 0.2, 'needed', 'none', 0.3),     # none too high
        row(4, 210, 1, 0.0, 'needed', 'blackboard', 0.55), row(4, 210, 1, 0.0, 'needed', 'none', 0.05)]   # gap 0.5, larger R
assert chooseClueSettings(rows) == dict(m=3, r=1, R=105, guess=0.0)
assert chooseClueSettings(rows[2:6]) is None
inc = sweepIncentives([row(3, 105, 1, 0.0, 'needed', 'blackboard', 0.6, welfare=0.8, sharerReward=5.0, nonSharerReward=7.0),
                              row(3, 105, 1, 0.0, 'never', 'blackboard', 0.2, welfare=0.3, sharerReward=float('nan'), nonSharerReward=2.0)],
                      dict(m=3, r=1, R=105, guess=0.0))
assert inc['welfareNeeded'] == 0.8 and inc['welfareNever'] == 0.3 and inc['sharerReward'] == 5.0

assert len(clueHash()) == 12 and clueHash({**CLUE_PLAN, 'version': 2}) != clueHash()
q = clueQuick()
assert all(c['cfg']['turns'] <= 10 for c in q) and {c['study'] for c in q} >= {'cluePilot', 'clue', 'clueFraming', 'clueLiar', 'clueNumpy', 'clueSweep'}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()